In [ ]:
import os
import gc
import pandas as pd
import numpy as np
import seaborn as sns
from scipy import stats
from scipy.stats import pearsonr
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.utils import resample
from matplotlib import pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Configuration
folds = range(0, 5)
stacking_path = '/UK_BB/brainbody/stacking'
commonality_path = '/UK_BB/brainbody/commonality'
os.makedirs(commonality_path, exist_ok=True)

In [ ]:
# Define helper functions 
def commonality_analysis(features_g_pred, target_g_real):
    model = LinearRegression()
    if len(features_g_pred.shape) < 2:
        features = features_g_pred.values.reshape(-1, 1)
    else:
        features = features_g_pred
    model.fit(features, target_g_real)
    return round(model.score(features, target_g_real), 4)

def decompose_variance(row):
    """Calculate variance components for a single row"""
    u_brain = row['R2_combined'] - row['R2_body']
    u_body = row['R2_combined'] - row['R2_brain']
    common = row['R2_brain'] + row['R2_body'] - row['R2_combined']
    
    return {
        'Unique variance: brain': round(u_brain, 4),
        'Unique variance: body': round(u_body, 4),
        'Common variance': round(common, 4),
        'Perc unique brain': round((u_brain/row['R2_combined'])*100, 2) if row['R2_combined'] else 0,
        'Perc unique body': round((u_body/row['R2_combined'])*100, 2) if row['R2_combined'] else 0,
        'Perc common': round((common/row['R2_combined'])*100, 2) if row['R2_combined'] else 0,
        'g-brain explained by body': round((common/(u_brain+common))*100, 2) if (u_brain+common) else 0,
        'g-body explained by brain': round((common/(u_body+common))*100, 2) if (u_body+common) else 0
    }


In [ ]:
# Define brain and body modalities
###############################################################
modalities_brain_body = [
'immune',
'renalhepatic',
'metabolic',
'cardiopulmonary',
'musculoskeletal',
'bone_densitometry',
'pwa',
'heart_mri',
'carotid_ultrasound',
'arterial_stiffness',
'ecg_rest',
'body_composition_by_impedance',
'body_composition_dxa',
'bone_dxa',
'kidneys_mri',
'liver_mri',
'abdominal_composition_mri_18_vars', #17 vars
'abdominal_organ_composition_mri_13_vars', #12 vars
'hearing',

'struct_fast',
'struct_sub_first',
'struct_fs_aseg_mean_intensity',
'struct_fs_aseg_volume',
'struct_ba_exvivo_area', 
'struct_ba_exvivo_mean_thickness',
'struct_ba_exvivo_volume',
'struct_a2009s_area',
'struct_a2009s_mean_thickness',
'struct_a2009s_volume',
'struct_dkt_area',
'struct_dkt_mean_thickness',
'struct_dkt_volume',
'struct_desikan_gw',
'struct_desikan_pial',
'struct_desikan_white_area',
'struct_desikan_white_mean_thickness',
'struct_desikan_white_volume',
'struct_subsegmentation',
'add_t1',
'add_t2',

"dwi_FA_tbss", "dwi_FA_prob",
"dwi_MD_tbss", "dwi_MD_prob",
"dwi_L1_tbss", "dwi_L1_prob",
"dwi_L2_tbss", "dwi_L2_prob",
"dwi_L3_tbss", "dwi_L3_prob",
"dwi_MO_tbss", "dwi_MO_prob",
"dwi_OD_tbss", "dwi_OD_prob",
"dwi_ICVF_tbss", "dwi_ICVF_prob",
"dwi_ISOVF_tbss", "dwi_ISOVF_prob",

'aparc_Tian_S1_FA_i2',
'aparc_Tian_S1_Length_i2',
'aparc_Tian_S1_SIFT2_FBC_i2',
'aparc_Tian_S1_Streamline_Count_i2',

'aparc_a2009s_Tian_S1_FA_i2',
'aparc_a2009s_Tian_S1_Length_i2',
'aparc_a2009s_Tian_S1_SIFT2_FBC_i2',
'aparc_a2009s_Tian_S1_Streamline_Count_i2',

'Glasser_Tian_S1_FA_i2',
'Glasser_Tian_S1_Length_i2',
'Glasser_Tian_S1_SIFT2_FBC_i2',
'Glasser_Tian_S1_Streamline_Count_i2',

'Glasser_Tian_S4_FA_i2',
'Glasser_Tian_S4_Length_i2',
'Glasser_Tian_S4_SIFT2_FBC_i2',
'Glasser_Tian_S4_Streamline_Count_i2',

'Schaefer7n200p_Tian_S1_FA_i2',
'Schaefer7n200p_Tian_S1_Length_i2',
'Schaefer7n200p_Tian_S1_SIFT2_FBC_i2',
'Schaefer7n200p_Tian_S1_Streamline_Count_i2',

'Schaefer7n1000p_Tian_S4_FA_i2',
'Schaefer7n1000p_Tian_S4_Length_i2',
'Schaefer7n1000p_Tian_S4_SIFT2_FBC_i2',
'Schaefer7n1000p_Tian_S4_Streamline_Count_i2',

"amplitudes_21",
"full_correlation_21",
"partial_correlation_21",
"amplitudes_55",
"full_correlation_55",
"partial_correlation_55",
'full_correlation_aparc_a2009s_Tian_S1',
'full_correlation_aparc_Tian_S1',
'full_correlation_Glasser_Tian_S1',
'full_correlation_Glasser_Tian_S4',
'full_correlation_Schaefer7n200p_Tian_S1',
'full_correlation_Schaefer7n500p_Tian_S4',
'partial_correlation_aparc_a2009s_Tian_S1',
'partial_correlation_aparc_Tian_S1',
'partial_correlation_Glasser_Tian_S1',
'partial_correlation_Glasser_Tian_S4',
'partial_correlation_Schaefer7n200p_Tian_S1',
'partial_correlation_Schaefer7n500p_Tian_S4'
]

modalities_brain = [
'struct_fast',
'struct_sub_first',
'struct_fs_aseg_mean_intensity',
'struct_fs_aseg_volume',
'struct_ba_exvivo_area', 
'struct_ba_exvivo_mean_thickness',
'struct_ba_exvivo_volume',
'struct_a2009s_area',
'struct_a2009s_mean_thickness',
'struct_a2009s_volume',
'struct_dkt_area',
'struct_dkt_mean_thickness',
'struct_dkt_volume',
'struct_desikan_gw',
'struct_desikan_pial',
'struct_desikan_white_area',
'struct_desikan_white_mean_thickness',
'struct_desikan_white_volume',
'struct_subsegmentation',
'add_t1',
'add_t2',

"dwi_FA_tbss", "dwi_FA_prob",
"dwi_MD_tbss", "dwi_MD_prob",
"dwi_L1_tbss", "dwi_L1_prob",
"dwi_L2_tbss", "dwi_L2_prob",
"dwi_L3_tbss", "dwi_L3_prob",
"dwi_MO_tbss", "dwi_MO_prob",
"dwi_OD_tbss", "dwi_OD_prob",
"dwi_ICVF_tbss", "dwi_ICVF_prob",
"dwi_ISOVF_tbss", "dwi_ISOVF_prob",

'aparc_Tian_S1_FA_i2',
'aparc_Tian_S1_Length_i2',
'aparc_Tian_S1_SIFT2_FBC_i2',
'aparc_Tian_S1_Streamline_Count_i2',

'aparc_a2009s_Tian_S1_FA_i2',
'aparc_a2009s_Tian_S1_Length_i2',
'aparc_a2009s_Tian_S1_SIFT2_FBC_i2',
'aparc_a2009s_Tian_S1_Streamline_Count_i2',

'Glasser_Tian_S1_FA_i2',
'Glasser_Tian_S1_Length_i2',
'Glasser_Tian_S1_SIFT2_FBC_i2',
'Glasser_Tian_S1_Streamline_Count_i2',

'Glasser_Tian_S4_FA_i2',
'Glasser_Tian_S4_Length_i2',
'Glasser_Tian_S4_SIFT2_FBC_i2',
'Glasser_Tian_S4_Streamline_Count_i2',

'Schaefer7n200p_Tian_S1_FA_i2',
'Schaefer7n200p_Tian_S1_Length_i2',
'Schaefer7n200p_Tian_S1_SIFT2_FBC_i2',
'Schaefer7n200p_Tian_S1_Streamline_Count_i2',

'Schaefer7n1000p_Tian_S4_FA_i2',
'Schaefer7n1000p_Tian_S4_Length_i2',
'Schaefer7n1000p_Tian_S4_SIFT2_FBC_i2',
'Schaefer7n1000p_Tian_S4_Streamline_Count_i2',

"amplitudes_21",
"full_correlation_21",
"partial_correlation_21",
"amplitudes_55",
"full_correlation_55",
"partial_correlation_55",
'full_correlation_aparc_a2009s_Tian_S1',
'full_correlation_aparc_Tian_S1',
'full_correlation_Glasser_Tian_S1',
'full_correlation_Glasser_Tian_S4',
'full_correlation_Schaefer7n200p_Tian_S1',
'full_correlation_Schaefer7n500p_Tian_S4',
'partial_correlation_aparc_a2009s_Tian_S1',
'partial_correlation_aparc_Tian_S1',
'partial_correlation_Glasser_Tian_S1',
'partial_correlation_Glasser_Tian_S4',
'partial_correlation_Schaefer7n200p_Tian_S1',
'partial_correlation_Schaefer7n500p_Tian_S4'
]

modalities_body = [
'immune',
'renalhepatic',
'metabolic',
'cardiopulmonary',
'musculoskeletal',
'bone_densitometry',
'pwa',
'heart_mri',
'carotid_ultrasound',
'arterial_stiffness',
'ecg_rest',
'body_composition_by_impedance',
'body_composition_dxa',
'bone_dxa',
'kidneys_mri',
'liver_mri',
'abdominal_composition_mri_18_vars', #17 vars
'abdominal_organ_composition_mri_13_vars', #12 vars
'hearing'
]

modalities_behav = [
'activity_touchscreen',
'accelerometry',
'activity_MET',
'activity_byrecall_i3',
'sleep_ac',
'localenv',
'diet',
'electronic_device_use',
'alcohol',
'sun_i0',
'sexual_factors',
'smoking'
]

In [ ]:
# Define modality renaming dictionary (modality_map)
modality_names = {
'hearing': 'Hearing',
'immune': 'Immune',
'renalhepatic': 'Renal & Hepatic',
'metabolic': 'Metabolic',
'cardiopulmonary': 'Cardiopulmonary',
'musculoskeletal': 'Musculoskeletal',
'bone_densitometry': 'Bone Densitometry of Heel',
'pwa': 'Pulse Wave Analysis',
'heart_mri': 'Heart MRI',
'carotid_ultrasound': 'Carotid Ultrasound',
'arterial_stiffness': 'Arterial Stiffness',
'ecg_rest': 'ECG at Rest',
'body_composition_by_impedance': 'Body Composition by Impedance',
'body_composition_dxa': 'Body composition by DXA',
'bone_dxa': 'Bone size, mineral and density by DXA',
'kidneys_mri': 'Kidney MRI',
'liver_mri': 'Liver MRI',
'abdominal_composition_mri_18_vars': 'Abdominal Composition by MRI',
'abdominal_organ_composition_mri_13_vars': 'Abdominal Organ Composition by MRI',
'struct_fast' : 'Regional grey matter volumes (FSL FAST)',
'struct_sub_first': 'Subcortical volumes (FSL FIRST)',

'struct_fs_aseg_mean_intensity' : 'ASEG Mean Intensity',
'struct_fs_aseg_volume' : 'ASEG Volume',


'struct_ba_exvivo_area' : 'BA ex-vivo Area',
'struct_ba_exvivo_mean_thickness' : 'BA ex-vivo Mean Thickness',
'struct_ba_exvivo_volume' : 'BA ex-vivo Volume',

'struct_a2009s_area' : 'a2009s Area',
'struct_a2009s_mean_thickness' : 'a2009s Mean Thickness',
'struct_a2009s_volume' : 'a2009s Volume',


'struct_dkt_area' : 'Desikan-Killiany-Tourville Area',
'struct_dkt_mean_thickness' : 'Desikan-Killiany-Tourville Mean Thickness',
'struct_dkt_volume' : 'Desikan-Killiany-Tourville Volume',


'struct_desikan_gw' : 'Desikan Grey/White Matter Contrast',
'struct_desikan_pial' : 'Desikan Pial',

'struct_desikan_white_area' : 'Desikan White Matter Area',
'struct_desikan_white_mean_thickness' : 'Desikan White Matter Mean Thickness',
'struct_desikan_white_volume' : 'Desikan White Matter Volume',
"struct_subsegmentation":'Subcortical Volumetric Subsegmentation',

'add_t1' : 'Whole-brain T1w',
'add_t2' : 'Whole-brain T2w',
"dwi_FA_tbss": "FA TBSS",
"dwi_FA_prob": "FA Prob.",
"dwi_MD_tbss": "MD TBSS",
"dwi_MD_prob": "MD Prob.",
"dwi_L1_tbss": "L1 TBSS",
"dwi_L1_prob": "L1 Prob.",
"dwi_L2_tbss": "L2 TBSS",
"dwi_L2_prob": "L2 Prob.",
"dwi_L3_tbss": "L3 TBSS",
"dwi_L3_prob": "L3 Prob.",
"dwi_MO_tbss": "MO TBSS",
"dwi_MO_prob": "MO Prob.",
"dwi_OD_tbss": "OD TBSS",
"dwi_OD_prob": "OD Prob.",
"dwi_ICVF_tbss": "ICVF TBSS",
"dwi_ICVF_prob": "ICVF Prob.",
"dwi_ISOVF_tbss": "ISOVF TBSS",
"dwi_ISOVF_prob": "ISOVF Prob.",
"amplitudes_21": " 21 IC amplitudes",
"amplitudes_55": "55 IC amplitudes",
"full_correlation_21": "21 IC Full corr.",
"full_correlation_55": "55 IC Full corr.",
"partial_correlation_21": " 21 IC Partial Corr.",
"partial_correlation_55": " 55 IC Partial Corr.",
# aparc Tian S1 (I)
'aparc_Tian_S1_FA_i2': 'aparc-I FA',
'aparc_Tian_S1_Length_i2': 'aparc-I Length',
'aparc_Tian_S1_SIFT2_FBC_i2': 'aparc-I SIFT2 FBC',
'aparc_Tian_S1_Streamline_Count_i2': 'aparc-I Streamline Count',

# aparc a2009s Tian S1 (I)
'aparc_a2009s_Tian_S1_FA_i2': 'aparc.a2009s-I FA',
'aparc_a2009s_Tian_S1_Length_i2': 'aparc.a2009s-I Length',
'aparc_a2009s_Tian_S1_SIFT2_FBC_i2': 'aparc.a2009s-I SIFT2 FBC',
'aparc_a2009s_Tian_S1_Streamline_Count_i2': 'aparc.a2009s-I Streamline Count',

# Glasser Tian S1 (I)
'Glasser_Tian_S1_FA_i2': 'Glasser-I FA',
'Glasser_Tian_S1_Length_i2': 'Glasser-I Length',
'Glasser_Tian_S1_SIFT2_FBC_i2': 'Glasser-I SIFT2 FBC',
'Glasser_Tian_S1_Streamline_Count_i2': 'Glasser-I Streamline Count',

# Glasser Tian S4 (IV)
'Glasser_Tian_S4_FA_i2': 'Glasser-IV FA',
'Glasser_Tian_S4_Length_i2': 'Glasser-IV Length',
'Glasser_Tian_S4_SIFT2_FBC_i2': 'Glasser-IV SIFT2 FBC',
'Glasser_Tian_S4_Streamline_Count_i2': 'Glasser-IV Streamline Count',

# Schaefer7n1000p Tian S4 (IV) (in reality: Schaefer7n200p Tian S1)
'Schaefer7n1000p_Tian_S4_FA_i2': 'Schaefer7n200p-I FA', #'Schaefer7n1000p-IV FA',
'Schaefer7n1000p_Tian_S4_Length_i2': 'Schaefer7n200p-I Length',#'Schaefer7n1000p-IV Length',
'Schaefer7n1000p_Tian_S4_SIFT2_FBC_i2': 'Schaefer7n200p-I SIFT2 FBC',#'Schaefer7n1000p-IV SIFT2 FBC',
'Schaefer7n1000p_Tian_S4_Streamline_Count_i2': 'Schaefer7n200p-I Streamline Count', #'Schaefer7n1000p-IV Streamline Count'

# Schaefer7n200p Tian S4 (IV) (in reality: Schaefer7n500p Tian S4)
'Schaefer7n200p_Tian_S1_FA_i2': 'Schaefer7n500p-IV FA',
'Schaefer7n200p_Tian_S1_Length_i2': 'Schaefer7n500p-IV Length',
'Schaefer7n200p_Tian_S1_SIFT2_FBC_i2': 'Schaefer7n500p-IV SIFT2 FBC',
'Schaefer7n200p_Tian_S1_Streamline_Count_i2': 'Schaefer7n500p-IV Streamline Count',

# Schaefer7n500p Tian S4 (IV) (in reality: Schaefer7n1000p Tian S4)
'Schaefer7n500p_Tian_S4_FA_i2': 'Schaefer7n1000p-IV FA',
'Schaefer7n500p_Tian_S4_Length_i2': 'Schaefer7n1000p-IV Length',
'Schaefer7n500p_Tian_S4_SIFT2_FBC_i2': 'Schaefer7n1000p-IV SIFT2 FBC',
'Schaefer7n500p_Tian_S4_Streamline_Count_i2': 'Schaefer7n1000p-IV Streamline Count',

# Resting state 
'full_correlation_aparc_a2009s_Tian_S1' : 'aparc.a2009s-I Full Corr.',
'full_correlation_aparc_Tian_S1': 'aparc-I Full Corr.',
'full_correlation_Glasser_Tian_S1': 'Glasser-I Full Corr.',
'full_correlation_Glasser_Tian_S4': 'Glasser-IV Full Corr.',
'full_correlation_Schaefer7n200p_Tian_S1': 'Schaefer7n200p-I Full Corr.',
'full_correlation_Schaefer7n500p_Tian_S4': 'Schaefer7n500p-IV Full Corr.',
'partial_correlation_aparc_a2009s_Tian_S1': 'aparc.a2009s-I Partial Corr.',
'partial_correlation_aparc_Tian_S1': 'aparc-I Partial Corr.',
'partial_correlation_Glasser_Tian_S1': 'Glasser-I Partial Corr.',
'partial_correlation_Glasser_Tian_S4': 'Glasser-IV Partial Corr.',
'partial_correlation_Schaefer7n200p_Tian_S1': 'Schaefer7n200p-I Partial Corr.',
'partial_correlation_Schaefer7n500p_Tian_S4': 'Schaefer7n500p-IV Partial Corr.',


'activity_touchscreen' :'Activity: Daily behaviour',
'accelerometry':'Activity: Accelerometry',
'activity_MET':'Activity: MET',
'activity_byrecall_i3':'Activity: Online (by recall)',
'sleep_ac':'Sleep',
'localenv':'Local environment',
'diet':'Diet',
'electronic_device_use':'Electronic device use',
'alcohol':'Alcohol',
'sun_i0':'Sun exposure',
'sexual_factors':'Sexual Factors',
'smoking':'Smoking',

'lifestyle-envir': 'Lifestyle & Environment',

'allmri': '3 Brain MRI Modalities Stacked',
'dwi': 'Brain dwMRI Stacked',
'smri': 'Brain sMRI Stacked',
'rs': 'Brain rsMRI Stacked',
'body': 'Body Physiology Stacked',
'lifestyle-envir': 'Lifestyle & Environment',
'brain-plus-body': '3 Brain MRI Modalities & Body Stacked',
'brain-body': 'Brain & Body Stacked',
'body-only': 'Body Physiology Stacked',
'all': 'Behaviour, body, and brain stacked'
}

In [ ]:
# Demographics confounds
demographics = pd.read_csv('/UK_BB/brainbody/demographics_full.csv')
# Rename columns and count NAs
demo = demographics[[
'eid',
'31-0.0',
'21000-0.0',
'21003-2.0']].rename(columns={
'31-0.0':'Sex',
'21000-0.0':'Ethnicity',
'21003-2.0':'Age'})

# 1. Composite lifestyle-environment explained by composite body and brain

In [ ]:
# Combine data across folds and compute R2
folds = range(0, 5)
stacking_path = '/UK_BB/brainbody/stacking'
commonality_path = '/UK_BB/brainbody/commonality/behav'
os.makedirs(commonality_path, exist_ok=True)

modalities_to_compare = {
    'allmri': ('brain', 'allmri'),
    'dwi': ('brain', 'dwi'),
    'rs': ('brain', 'rs'),
    'smri': ('brain', 'smri'),
    'body': ('body', None),
    'brain-body': ('brain-body', None)
}

r2_results = []

for comp_name, (folder, subfolder) in modalities_to_compare.items():
    df_concat = []
    
    for fold in folds:
        # Load lifestyle-envir predictions (same for all)
        behav_path = os.path.join(stacking_path, 'lifestyle-envir', 'folds', 
                                 f'fold_{fold}', 'g_pred',
                                 f'lifestyle-envir_target_pred_2nd_level_0_outer_test_fold_{fold}.csv')
        behav_pred = pd.read_csv(behav_path)[['eid', 'g_pred_stack_test']].rename(
            columns={'g_pred_stack_test': 'g_pred_behav_test'})
        
        # Load comparison modality predictions:
        if comp_name == 'brain-body':
            # 1: brain-body (no subfolder)
            comp_path = os.path.join(stacking_path, 'brain-body', 'folds', 
                                   f'fold_{fold}', 'g_pred',
                                   f'brain-body_target_pred_2nd_level_0_outer_test_fold_{fold}.csv')
        
        elif comp_name == 'body':
            # 2: body
            comp_path = os.path.join(stacking_path, 'body', 'folds',
                                   f'fold_{fold}', 'g_pred',
                                   f'body_target_pred_2nd_level_0_outer_test_fold_{fold}.csv')
        
        else:
            # 3: brain submodalities (allmri, dwi, rs, smri)
            comp_path = os.path.join(stacking_path, folder, subfolder, 'folds',
                                   f'fold_{fold}', 'g_pred',
                                   f'{subfolder}_target_pred_2nd_level_rf_test_fold_{fold}.csv')
        
        comp_pred = pd.read_csv(comp_path)[['eid', 'g_pred_stack_test']].rename(
            columns={'g_pred_stack_test': f'g_pred_{comp_name}_test'})
        
        # Load observed g
        obs_path = os.path.join(stacking_path, 'lifestyle-envir',
                              'features_test_level1_stacked_outer',
                              f'features_test_level1_outer_g_matched_fold_{fold}.csv')
        g_obs = pd.read_csv(obs_path)[['eid', 'g']].rename(columns={'g': 'g_obs_test'})
        
        # Merge
        merged = behav_pred.merge(comp_pred, on='eid').merge(g_obs, on='eid')
        df_concat.append(merged.drop(columns=['eid']))
        
    # Combine folds
    all_g = pd.concat(df_concat, axis=0, ignore_index=True).dropna()
    
    # Save
    output_name = f'g_predictions_behav_vs_{comp_name}'
    all_g.to_csv(os.path.join(commonality_path, f'{output_name}.csv'), index=False)
    
    # Calculate R2
    r2_behav = commonality_analysis(all_g['g_pred_behav_test'], all_g['g_obs_test'])
    r2_comp = commonality_analysis(all_g[f'g_pred_{comp_name}_test'], all_g['g_obs_test'])
    r2_combined = commonality_analysis(
            pd.concat([all_g['g_pred_behav_test'], 
                      all_g[f'g_pred_{comp_name}_test']], axis=1),
            all_g['g_obs_test']
        )
    
    r2_results.append({
        'Body and brain phenotypes and markers': f'behav_vs_{comp_name}',
        'Behavioural domain': 'lifestyle-envir',
        'R2 behaviour': r2_behav,
        'R2 brain/body': r2_comp,
        'R2 combined': r2_combined,
        'N': len(all_g)
    })

# Save results
r2 = pd.DataFrame(r2_results)
r2.to_csv(os.path.join(commonality_path, 'r2_behav_vs_all_composite.csv'), index=False)

print("\nR2 Results:")
print(r2)

In [ ]:
# Decompose variance
r2_behav_vs_all_composite = pd.read_csv(os.path.join(commonality_path, 'r2_behav_vs_all_composite.csv'))
r2_decompose = r2_behav_vs_all_composite.copy()

for idx, row in r2_decompose.iterrows():
    
    # Calculate variance components
    u_behav = row['R2 combined'] - row['R2 brain/body']
    u_brain_body = row['R2 combined'] - row['R2 behaviour']
    common = row['R2 behaviour'] + row['R2 brain/body'] - row['R2 combined']
    
    total = row['R2 combined']

    # Calculate percentages
    perc_behav = round((u_behav / total) * 100, 2) if total != 0 else 0
    perc_brain_body = round((u_brain_body / total) * 100, 2) if total != 0 else 0
    perc_common = round((common / total) * 100, 2) if total != 0 else 0

    # Calculate proportions
    prop_behav_expl_by_brain_body = round((common / (u_behav + common)) * 100, 2) if (u_behav + common) != 0 else 0
    prop_brain_body_exp_by_behav = round((common / (u_brain_body + common)) * 100, 2) if (u_brain_body + common) != 0 else 0

    # Store results
    r2_decompose.at[idx, 'Unique variance: behaviour'] = round(u_behav, 4)
    r2_decompose.at[idx, 'Unique variance: brain/body'] = round(u_brain_body, 4)
    r2_decompose.at[idx, 'Common variance'] = round(common, 4)
    r2_decompose.at[idx, 'Perc unique behaviour'] = perc_behav
    r2_decompose.at[idx, 'Perc unique brain/body'] = perc_brain_body
    r2_decompose.at[idx, 'Perc common'] = perc_common
    r2_decompose.at[idx, 'g-behaviour explained by brain/body'] = round(prop_behav_expl_by_brain_body, 4)
    r2_decompose.at[idx, 'g-brain/body explained by behaviour'] = round(prop_brain_body_exp_by_behav, 4)


    # Print
    print('____________________')
    print('R2 behaviour:', round(row['R2 behaviour'], 4))
    print('R2 brain/body:', round(row['R2 brain/body'], 4))
    print('R2 combined:', round(row['R2 combined'], 4))
    print('____________________')
    print(f"Unique to behaviour: {round(u_behav, 4):.4f} ({perc_behav:.2f}%)")
    print(f"Unique to brain/body: {round(u_brain_body, 4):.4f} ({perc_brain_body:.2f}%)")
    print(f"Common variance: {round(common, 4):.4f} ({perc_common:.2f}%)")
    print('____________________')
    print(f"Proportion of behavioural variance explained by brain/body: {prop_behav_expl_by_brain_body:.2f}%")
    print(f"Proportion of brain/body variance explained by behaviour: {prop_brain_body_exp_by_behav:.2f}%")

# Save
r2_decompose.to_csv(os.path.join(commonality_path, 'commonality_results_decompos_behav_vs_brain-body_stacked.csv'), index=False)

r2_decompose.to_excel(
    os.path.join(commonality_path, f'commonality_results_decompos_behav_vs_brain-body_stacked.xlsx'),
    index=False,
    engine='openpyxl'
)

r2_decompose.sort_values(by='g-behaviour explained by brain/body', ascending=False).to_excel(
    os.path.join(commonality_path, f'commonality_results_decompos_behav_vs_brain-body_stacked_sorted.xlsx'),
    index=False,
    engine='openpyxl'
)

In [ ]:
# Rename modalities in excel
name_mappings = {
    'behav_vs_allmri': 'Brain phenotypes stacked (composite brain marker)',
    'behav_vs_dwi': 'Brain dwMRI Stacked',
    'behav_vs_smri': 'Brain sMRI stacked',
    'behav_vs_rs': 'Brain rsMRI stacked',
    'behav_vs_body': 'Body phenotypes stacked (composite body marker)',
    'behav_vs_brain-body': 'Brain & body stacked',
    'lifestyle-envir': 'Behaviour, lifestyle, and environment stacked (composite behavioural marker)',
}

for file_suffix in ['', '_sorted']:
    file_path = os.path.join(commonality_path, f'commonality_results_decompos_behav_vs_brain-body_stacked{file_suffix}.xlsx')
    
    # Load the Excel file
    df = pd.read_excel(file_path)
    
    # Apply name mappings to the relevant columns
    df['Body and brain phenotypes and markers'] = df['Body and brain phenotypes and markers'].replace(name_mappings)
    df['Behavioural domain'] = df['Behavioural domain'].replace(name_mappings)
    
    # Save back to the same file
    df.to_excel(os.path.join(commonality_path, f'commonality_results_decompos_behav_vs_brain-body_stacked{file_suffix}_renamed.xlsx'), index=False, engine='openpyxl')

print("Excel files have been updated with the new column names.")

# 2. Lifestyle/envirnment phenotypes explained by composite body and brain

In [ ]:
# Combine data across folds and compute R2
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# Configuration
folds = range(0, 5)
stacking_path =  '/UK_BB/brainbody/stacking'
base_path = '/UK_BB/brainbody'
commonality_path = '/UK_BB/brainbody/commonality/behav'
os.makedirs(commonality_path, exist_ok=True)

r2_results = []

for behav_mod in modalities_behav:
    for comp_name, (folder, subfolder) in modalities_to_compare.items():
        
        df_concat = []
        
        for fold in folds:
            
            # Load behavioural domain prediction
            behav_pred_path = os.path.join(
                base_path,
                'lifestyle-envir-body',
                'folds', f'fold_{fold}', 'g_pred',
                f'{behav_mod}_g_pred_XGB_test_with_id_fold_{fold}.csv'
            )
            
            behav_pred = pd.read_csv(behav_pred_path).rename(
                columns={f'g_pred_test_{behav_mod}': f'g_pred_behav_{behav_mod}_test'}
            )
            
            # Load comparison modality prediction
            if comp_name == 'brain-body':
                comp_path = os.path.join(
                    stacking_path, 'brain-body', 'folds', f'fold_{fold}', 'g_pred',
                    f'brain-body_target_pred_2nd_level_0_outer_test_fold_{fold}.csv'
                )
            elif comp_name == 'body':
                comp_path = os.path.join(
                    stacking_path, 'body', 'folds', f'fold_{fold}', 'g_pred',
                    f'body_target_pred_2nd_level_0_outer_test_fold_{fold}.csv'
                )
            else:
                comp_path = os.path.join(
                    stacking_path, folder, subfolder, 'folds', f'fold_{fold}', 'g_pred',
                    f'{subfolder}_target_pred_2nd_level_rf_test_fold_{fold}.csv'
                )
            
            comp_pred = pd.read_csv(comp_path).rename(
                columns={'g_pred_stack_test': f'g_pred_{comp_name}_test'}
            )
            
            # Load observed g for this behavioural domain
            obs_path = os.path.join(
                stacking_path,
                'lifestyle-envir',
                'features_test_level1_stacked_outer',
                f'features_test_level1_outer_g_matched_fold_{fold}.csv'
            )
            
            g_obs = pd.read_csv(obs_path)[['eid', 'g']].rename(
                columns={'g': 'g_obs_test'}
            )
            
            # Merge
            merged = (
                behav_pred
                .merge(comp_pred, on='eid')
                .merge(g_obs, on='eid')
                .drop(columns=['eid'])
            )
            
            df_concat.append(merged)
        
        # Combine folds
        all_g = pd.concat(df_concat, axis=0, ignore_index=True).dropna()
        
        # Save merged dataset
        output_name = f'g_obs_pred_{behav_mod}_vs_{comp_name}.csv'
        all_g.to_csv(os.path.join(commonality_path, output_name), index=False)
        
        # Compute R2
        r2_behav = commonality_analysis(all_g[f'g_pred_behav_{behav_mod}_test'], all_g['g_obs_test'])
        r2_comp = commonality_analysis(all_g[f'g_pred_{comp_name}_test'], all_g['g_obs_test'])
        r2_combined = commonality_analysis(
            pd.concat([
                all_g[f'g_pred_behav_{behav_mod}_test'],
                all_g[f'g_pred_{comp_name}_test']
            ], axis=1),
            all_g['g_obs_test']
        )
        
        r2_results.append({
            'Body and brain phenotypes and markers': comp_name,
            'Behavioural domain': behav_mod,
            'R2 behaviour': r2_behav,
            'R2 brain/body': r2_comp,
            'R2 combined': r2_combined,
            'N': len(all_g)
        })
# Save results
r2 = pd.DataFrame(r2_results)
r2.to_csv(os.path.join(commonality_path, 'r2_behav_individual_vs_brain-body_composite.csv'), index=False)

print("\nR2 Results:")
print(r2)


In [ ]:
# Decompose variance
r2_behav_individual_vs_all_composite = pd.read_csv(os.path.join(commonality_path, 'r2_behav_individual_vs_brain-body_composite.csv'))
r2_decompose = r2_behav_individual_vs_all_composite.copy()

for idx, row in r2_decompose.iterrows():
    
    # Calculate variance components
    u_behav = row['R2 combined'] - row['R2 brain/body']
    u_brain_body = row['R2 combined'] - row['R2 behaviour']
    common = row['R2 behaviour'] + row['R2 brain/body'] - row['R2 combined']
    
    total = row['R2 combined']

    # Calculate percentages
    perc_behav = round((u_behav / total) * 100, 2) if total != 0 else 0
    perc_brain_body = round((u_brain_body / total) * 100, 2) if total != 0 else 0
    perc_common = round((common / total) * 100, 2) if total != 0 else 0

    # Calculate proportions
    prop_behav_expl_by_brain_body = round((common / (u_behav + common)) * 100, 2) if (u_behav + common) != 0 else 0
    prop_brain_body_exp_by_behav = round((common / (u_brain_body + common)) * 100, 2) if (u_brain_body + common) != 0 else 0

    # Store results
    r2_decompose.at[idx, 'Unique variance: behaviour'] = round(u_behav, 4)
    r2_decompose.at[idx, 'Unique variance: brain/body'] = round(u_brain_body, 4)
    r2_decompose.at[idx, 'Common variance'] = round(common, 4)
    r2_decompose.at[idx, 'Perc unique behaviour'] = perc_behav
    r2_decompose.at[idx, 'Perc unique brain/body'] = perc_brain_body
    r2_decompose.at[idx, 'Perc common'] = perc_common
    r2_decompose.at[idx, 'g-behaviour explained by brain/body'] = round(prop_behav_expl_by_brain_body, 4)
    r2_decompose.at[idx, 'g-brain/body explained by behaviour'] = round(prop_brain_body_exp_by_behav, 4)


    # Print
    print('____________________')
    print('R2 behaviour:', round(row['R2 behaviour'], 4))
    print('R2 brain/body:', round(row['R2 brain/body'], 4))
    print('R2 combined:', round(row['R2 combined'], 4))
    print('____________________')
    print(f"Unique to behaviour: {round(u_behav, 4):.4f} ({perc_behav:.2f}%)")
    print(f"Unique to brain/body: {round(u_brain_body, 4):.4f} ({perc_brain_body:.2f}%)")
    print(f"Common variance: {round(common, 4):.4f} ({perc_common:.2f}%)")
    print('____________________')
    print(f"Proportion of behavioural variance explained by brain/body: {prop_behav_expl_by_brain_body:.2f}%")
    print(f"Proportion of brain/body variance explained by behaviour: {prop_brain_body_exp_by_behav:.2f}%")

# Save
r2_decompose.to_csv(os.path.join(commonality_path, 'commonality_results_decompos_behav_individual_vs_brain-body_stacked.csv'), index=False)

r2_decompose.to_excel(
    os.path.join(commonality_path, f'commonality_results_decompos_behav_individual_vs_brain-body_stacked.xlsx'),
    index=False,
    engine='openpyxl'
)

r2_decompose.sort_values(by='g-behaviour explained by brain/body', ascending=False).to_excel(
    os.path.join(commonality_path, f'commonality_results_decompos_behav_individual_vs_brain-body_stacked_sorted.xlsx'),
    index=False,
    engine='openpyxl'
)

In [ ]:
# Rename modalities in excel
name_mappings = {
'allmri': 'Brain phenotypes stacked (composite brain marker)',
'dwi': 'Brain dwMRI Stacked',
'smri': 'Brain sMRI stacked',
'rs': 'Brain rsMRI stacked',
'body': 'Body phenotypes stacked (composite body marker)',
'brain-body': 'Brain & body stacked',
'lifestyle-envir': 'Behaviour, lifestyle, and environment stacked (composite behavioural marker)',
'activity_touchscreen' :'Activity: Daily behaviour',
'accelerometry':'Activity: Accelerometry',
'activity_MET':'Activity: MET',
'activity_byrecall_i3':'Activity: Online (by recall)',
'sleep_ac':'Sleep',
'localenv':'Local environment',
'diet':'Diet',
'electronic_device_use':'Electronic device use',
'alcohol':'Alcohol',
'sun_i0':'Sun exposure',
'sexual_factors':'Sexual Factors',
'smoking':'Smoking',
}

for file_suffix in ['', '_sorted']:
    file_path = os.path.join(commonality_path, f'commonality_results_decompos_behav_individual_vs_brain-body_stacked{file_suffix}.xlsx')
    
    # Load the Excel file
    df = pd.read_excel(file_path)
    
    # Apply name mappings to the relevant columns
    df['Body and brain phenotypes and markers'] = df['Body and brain phenotypes and markers'].replace(name_mappings)
    df['Behavioural domain'] = df['Behavioural domain'].replace(name_mappings)
    
    # Save back to the same file
    df.to_excel(os.path.join(commonality_path, f'commonality_results_decompos_behav_individual_vs_brain-body_stacked{file_suffix}_renamed.xlsx'), index=False, engine='openpyxl')

print("Excel files have been updated with the new column names.")

## 3. g-age explained by composite behaviour plus body plus brain - 3-var commonality

- U (i) = R2_ijk – R2_jk
- U (j) = R2_ijk – R2_ik
- U (k) = R2_ijk – R2_ij

- C (ij) = R2_ik + R2_jk – R2_k – R2_ijk
- C (ik) = R2_ij + R2_jk – R2_j – R2_ijk
- C (jk) = R2_ij + R2_ik – R2_i – R2_ijk
- C (ijk) = R2_i + R2_j + R2_k – R2_ij – R2_ik – R2_jk + R2_ijk

In [ ]:
# Commonality analysis
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# Configuration
folds = range(0, 5)
stacking_path = '/UK_BB/brainbody/stacking'
commonality_path = '/UK_BB/brainbody/commonality/behav/age'
os.makedirs(commonality_path, exist_ok=True)

# Load demographic data
demo_age = demo[['eid', 'Age']]

# Store concatenated data for each fold
df_concat = []

for fold in folds:
    print(f'\n############## FOLD {fold} ##############')
    
    # Get predictions paths
    pred_path_body_brain = os.path.join(
        stacking_path, 'brain-body', 'folds', f'fold_{fold}', 'g_pred',
        f'brain-body_target_pred_2nd_level_0_outer_test_fold_{fold}.csv')
    
    pred_path_behav = os.path.join(
        stacking_path, 'lifestyle-envir', 'folds', f'fold_{fold}', 'g_pred',
        f'lifestyle-envir_target_pred_2nd_level_0_outer_test_fold_{fold}.csv')
    
    # Load predictions
    pred_body_brain = pd.read_csv(pred_path_body_brain).rename(
        columns={'g_pred_stack_test': 'g_pred_body_brain'})
    
    pred_behav = pd.read_csv(pred_path_behav).rename(
        columns={'g_pred_stack_test': 'g_pred_behav'})
    
    # Get observations paths
    obs_path_body_brain = os.path.join(
        stacking_path, 'brain-body',
        'features_test_level1_stacked_outer', f'features_test_level1_outer_g_matched_fold_{fold}.csv')
    
    obs_path_behav = os.path.join(
        stacking_path, 'lifestyle-envir',
        'features_test_level1_stacked_outer', f'features_test_level1_outer_g_matched_fold_{fold}.csv')
    
    g_obs_body_brain = pd.read_csv(obs_path_body_brain)[['eid', 'g']].rename(columns={'g': 'g_obs'})
    g_obs_behav = pd.read_csv(obs_path_behav)[['eid', 'g']].rename(columns={'g': 'g_obs'})
    
    # Restrict to subjects present in both datasets
    g_obs = g_obs_body_brain.merge(g_obs_behav, on='eid', suffixes=('_body_brain', '_behav'))
    
    # Ensure g values match for all common subjects
    if not (g_obs['g_obs_body_brain'] == g_obs['g_obs_behav']).all():
        raise ValueError(f"Mismatch in g_obs values in fold {fold}")
    
    # Keep a single g_obs column
    g_obs = g_obs[['eid', 'g_obs_body_brain']].rename(columns={'g_obs_body_brain': 'g_obs'})

    # Merge all data including age
    all_g = (pred_body_brain.merge(pred_behav, on='eid')
             .merge(g_obs, on='eid')
             .merge(demo_age, on='eid')
             .drop(columns=['eid']))
    
    print('Merged data for this fold:')
    print('Shape:', all_g.shape)
    print('Columns:', all_g.columns.tolist())
    print('NA counts:', all_g.isna().sum().sum())
    
    df_concat.append(all_g)

# Check if there are any data
if not df_concat:
    print("No data loaded! Check file paths.")
    exit()

# Concatenate all folds
all_g_behav_brain_body_age = pd.concat(df_concat, axis=0, ignore_index=True)
all_g_behav_brain_body_age.to_csv(os.path.join(commonality_path, f'g_obs_pred_behav_plus_brain-body_plus_age.csv'), index=False)

print('Final shape:', all_g_behav_brain_body_age.shape)
print('Final columns:', all_g_behav_brain_body_age.columns.tolist())
print('NA counts:\n', all_g_behav_brain_body_age.isna().sum())

# Calculate R2 values for 3-variable commonality analysis
print('\n############## Calculating R2 ##############')

r2_behav = commonality_analysis(all_g_behav_brain_body_age['g_pred_behav'], all_g_behav_brain_body_age['g_obs'])
r2_bodybrain = commonality_analysis(all_g_behav_brain_body_age['g_pred_body_brain'], all_g_behav_brain_body_age['g_obs'])
r2_age = commonality_analysis(all_g_behav_brain_body_age['Age'], all_g_behav_brain_body_age['g_obs'])
r2_behav_bodybrain = commonality_analysis(
    pd.concat([all_g_behav_brain_body_age['g_pred_behav'], all_g_behav_brain_body_age['g_pred_body_brain']], axis=1), 
    all_g_behav_brain_body_age['g_obs']
)
r2_behav_age = commonality_analysis(
    pd.concat([all_g_behav_brain_body_age['g_pred_behav'], all_g_behav_brain_body_age['Age']], axis=1), 
    all_g_behav_brain_body_age['g_obs']
)
r2_bodybrain_age = commonality_analysis(
    pd.concat([all_g_behav_brain_body_age['g_pred_body_brain'], all_g_behav_brain_body_age['Age']], axis=1), 
    all_g_behav_brain_body_age['g_obs']
)
r2_behav_bodybrain_age = commonality_analysis(
    pd.concat([all_g_behav_brain_body_age['g_pred_behav'], all_g_behav_brain_body_age['g_pred_body_brain'], all_g_behav_brain_body_age['Age']], axis=1), 
    all_g_behav_brain_body_age['g_obs']
)

print('\nR2 Results:')
print('____________________')
print(f'R2 behaviour: {r2_behav:.4f}')
print(f'R2 brain/body: {r2_brainbody:.4f}')
print(f'R2 age: {r2_age:.4f}')
print(f'R2 behaviour + brain/body: {r2_behav_bodybrain:.4f}')
print(f'R2 behaviour + age: {r2_behav_age:.4f}')
print(f'R2 brain/body + age: {r2_bodybrain_age:.4f}')
print(f'R2 behaviour + brain/body + age: {r2_behav_bodybrain_age:.4f}')

# Calculate commonality components
print('\n############## Decomposition  ##############')

# Unique variances (U)
u_behav = r2_behav_bodybrain_age - r2_bodybrain_age
u_bodybrain = r2_behav_bodybrain_age - r2_behav_age
u_age = r2_behav_bodybrain_age - r2_behav_bodybrain

print("Unique variance for Body:", round((u_behav),4))
print("Unique variance for Brain:", round((u_bodybrain),4))
print("Unique variance for Age:", round((u_age),4))

# Common variances (C)
c_behav_bodybrain = r2_behav_age + r2_bodybrain_age - r2_age - r2_behav_bodybrain_age
c_behav_age = r2_behav_bodybrain + r2_bodybrain_age - r2_bodybrain - r2_behav_bodybrain_age
c_bodybrain_age = r2_behav_bodybrain + r2_behav_age - r2_behav - r2_behav_bodybrain_age
c_behav_bodybrain_age = r2_behav + r2_bodybrain + r2_age - r2_behav_bodybrain - r2_behav_age - r2_bodybrain_age + r2_behav_bodybrain_age

age_total = u_age + c_behav_age + c_behav_bodybrain_age + c_bodybrain_age

prop_age_expl_by_behav_only = (c_behav_age + c_behav_bodybrain_age) / (age_total)
prop_age_expl_by_bodybrain_only = (c_bodybrain_age + c_behav_bodybrain_age) / (age_total)
prop_age_expl_by_behav_OR_bodybrain = (c_behav_age + c_bodybrain_age + c_behav_bodybrain_age) / (age_total)
prop_age_expl_by_behav_AND_bodybrain = (c_behav_bodybrain_age) / (age_total)

print("Common variance for Body & Brain:", round((c_behav_bodybrain),4))
print("Common variance for Body & Age:", round((c_behav_age),4))
print("Common variance for Brain & Age:", round((c_bodybrain_age),4))
print("Common variance for Body, Brain & Age:", round((c_behav_bodybrain_age),4))


# Build a single-row decomposition table
decomp = pd.DataFrame([{
    # R2
    'R2 behaviour': round(r2_behav, 2),
    'R2 body/brain': round(r2_bodybrain, 4),
    'R2 age': round(r2_age, 4),
    'R2 behaviour + body/brain': round(r2_behav_bodybrain, 2),
    'R2 behaviour + age': round(r2_behav_age, 4),
    'R2 body/brain + age': round(r2_bodybrain_age, 4),
    'R2 behaviour + body/brain + age': round(r2_behav_bodybrain_age, 4),
    # Raw variances
    'Unique: behaviour': round(u_behav, 4),
    'Unique: body/brain': round(u_bodybrain, 4),
    'Unique: age': round(u_age, 4),
    'Common: behaviour & body/brain': round(c_behav_bodybrain, 4),
    'Common: behaviour & age': round(c_behav_age, 4),
    'Common: body/brain & age': round(c_bodybrain_age, 4),
    'Common: all': round(c_behav_bodybrain_age, 4),

    # Percentages
    'Perc unique behaviour': round(u_behav / r2_behav_bodybrain_age * 100, 2),
    'Perc unique body/brain': round(u_bodybrain / r2_behav_bodybrain_age * 100, 2),
    'Perc unique age': round(u_age / r2_behav_bodybrain_age * 100, 2),
    'Perc common behaviour-body/brain': round(c_behav_bodybrain / r2_behav_bodybrain_age * 100, 2),
    'Perc common behaviour-age': round(c_behav_age / r2_behav_bodybrain_age * 100, 2),
    'Perc common body/brain-age': round(c_bodybrain_age / r2_behav_bodybrain_age * 100, 2),
    'Perc common all': round(c_behav_bodybrain_age / r2_behav_bodybrain_age * 100, 2),

    # Additional interpretable quantities
    'g-age explained by behaviour': round(prop_age_expl_by_behav_only, 4)*100,
    'g-age explained by body/brain': round(prop_age_expl_by_bodybrain_only, 4)*100,
    'g-age explained by body/brain or behaviour': round(prop_age_expl_by_behav_OR_bodybrain, 4)*100,
    'g-age explained by body/brain & behaviour': round(prop_age_expl_by_behav_AND_bodybrain, 4)*100,

    # Total R2
    'Total R2 (behaviour+body/brain+age)': round(r2_behav_bodybrain_age, 4)}])

# Save results
decomp.to_csv(
    os.path.join(commonality_path, 'commonality_results_behav_plus_body-brain_age_3v.csv'),
    index=False
)

decomp.to_excel(
    os.path.join(commonality_path, 'commonality_results_behav_plus_body-brain_age_3v.xlsx'),
    index=False,
    engine='openpyxl'
)

print("\nFinal Results:")
print(decomp)

# Validate all components sum to total R2
components_sum = (u_behav + u_bodybrain + u_age + 
                  c_behav_bodybrain + c_behav_age + c_bodybrain_age + 
                  c_behav_bodybrain_age)

if abs(components_sum - r2_behav_bodybrain_age) > 0.000001:
    print(f"\nWARNING: Variance components don't sum to total R2!")
    print(f"Sum of components: {components_sum:.6f}")
    print(f"Total R2: {r2_behav_bodybrain_age:.6f}")
    print(f"Difference: {abs(components_sum - r2_behav_bodybrain_age):.6f}")
else:
    print(f"\n✓ Variance components correctly sum to total R2: {components_sum:.6f}")

## Combine tables: body and brain

Combine composite body and body phenotypes in one table

In [ ]:
# Combnie individual modalities and stacked: brain/body
ind = pd.read_excel(os.path.join(commonality_path, 'commonality_results_decompos_brain_vs_individual_body_sorted.xlsx'))
stack = pd.read_excel(os.path.join(commonality_path, f'commonality_results_decompos_brain_vs_body_stacked_sorted.xlsx'))
pd.concat([stack, ind], axis=0).sort_values(by='g-body explained by brain', ascending=False).to_excel(os.path.join(commonality_path, f'commonality_results_decompos_brain_vs_body_sorted_stack_ind_combined.xlsx'), index=False, engine='openpyxl')

Rename body/brain

In [ ]:
# Define renaming function
def process_modality_name(name):
    if pd.isna(name):
        return name
    
    # Handle specific replacements
    name = name.replace('Bone densitometry (of heel)', 'Bone Densitometry of Heel')
    name = name.replace('3 brain MRI modalities stacked', '3 Brain MRI Modalities Stacked')
    name = name.replace('stacked', 'Stacked')
    name = name.replace('Body physiology and composition Stacked', 'Body Physiology and Composition Stacked')
    name = name.replace('Bone size, mineral and density by DXA', 'Bone Size, Mineral and Density by DXA')
    name = name.replace('composition', 'Composition')
    
    
    # Capitalization - keep existing capitalization for all words
    #words = name.split()
    #processed_words = [word.capitalize() for word in words]
    
    #result = ' '.join(processed_words)
    return name

In [ ]:
# Correct phenotypes names and make them uniform
file_path = os.path.join(commonality_path, 'commonality_results_decompos_brain_vs_body_sorted_stack_ind_combined.xlsx')

df = pd.read_excel(file_path, engine='openpyxl')
# Apply the processing to the Modality column
df['Body modality'] = df['Body modality'].apply(process_modality_name)
df['Brain MRI modality'] = df['Brain MRI modality'].apply(process_modality_name)
#df = df.drop(columns='Type')
df = df[~df['Body modality'].isin(['Body composition stacked', 'Cardiopulmonary Stacked', 'Renal & Hepatic Stacked', 'Body Composition Stacked'])]
#df = df[df['Modality'] != 'Invalid Modality']

df.to_excel(os.path.join(commonality_path, f'commonality_results_decompos_brain_vs_body_sorted_stack_ind_combined_RENAMED.xlsx'), index=False, engine='openpyxl')
#f'commonality_results_decompos_brain_body_vs_age_sorted_stack_ind_combined_SORTED_BY_AGE_RENAMED.xlsx'
# Display the results to verify
print("Processed Modality column values:")
print(df['Brain MRI modality'].unique())
print(df['Body modality'].unique())

Rename age+body/brain

In [ ]:
# Define renaming function
def process_modality_name(name):
    if pd.isna(name):
        return name
    
    # Handle replacements
    name = name.replace('Corr.', 'Correlation').replace('Corr.', 'Correlation')
    name = name.replace('Prob.', 'Probabilistic')
    name = name.replace('(of heel)', 'of Heel')
    
    # Split into words and process each one
    words = name.split()
    processed_words = []
    
    for word in words:
        # Handle specific lowercase terms - but NOT 'iv' here since we need to handle it differently
        if word.lower() in ['and', 'aparc', 'by', 'at', 'ex-vivo', 'ex', 'vivo', 'of', 'heel', 'a2009s', 'aparc.a2009s']:
            processed_words.append(word.lower())
        # Keep specific MRI modalities as-is
        elif word in ['dwMRI', 'rsMRI', 'sMRI', 'MRI']:
            processed_words.append(word)
        # Handle hyphenated words separately to preserve 'iv' vs 'IV'
        elif '-' in word:
            parts = word.split('-')
            processed_parts = []
            for part in parts:
                if part.lower() in ['aparc', 'by', 'at', 'ex', 'vivo', 'of', 'heel', 'a2009s', 'and', 'aparc.a2009s']:
                    processed_parts.append(part.lower())
                elif part in ['dwMRI', 'rsMRI', 'sMRI', 'MRI', 'SIFT2', 'FBC', 'FA', 'TBSS', 'MD', 'MO', 'OD', 'ICVF', 'ISOVF', 'ECG', 'DXA', 'IV', 'I']:
                    processed_parts.append(part)
                elif (part.isupper() or 
                      (any(char.isupper() for char in part) and 
                       (any(char.isdigit() for char in part) or len(part) <= 4))):
                    processed_parts.append(part)
                else:
                    processed_parts.append(part.capitalize())
            processed_words.append('-'.join(processed_parts))
        # Keep acronyms and abbreviations
        elif (word.isupper() or 
              word in ['dwMRI', 'rsMRI', 'sMRI', 'MRI', 'SIFT2', 'FBC', 'FA', 'TBSS', 'MD', 'MO', 'OD', 'ICVF', 'ISOVF', 'ECG', 'DXA', 'IV', 'I'] or
              (any(char.isupper() for char in word) and 
               (any(char.isdigit() for char in word) or len(word) <= 4))):
            processed_words.append(word)
        # Capitalize all other words (regular words)
        else:
            processed_words.append(word.capitalize())
    
    result = ' '.join(processed_words)
    result = result.replace('Corr.', 'Correlation')
    return result

In [ ]:
# Correct phenotypes names and make them uniform
file_path = os.path.join(commonality_path, 'commonality_results_age_vs_brain_body_paired_male_female.xlsx')

df = pd.read_excel(file_path, engine='openpyxl')
# Apply the processing to the Modality column
df['Modality'] = df['Modality'].apply(process_modality_name)
#df = df.drop(columns='Type')
df = df[~df['Modality'].isin(['Body Composition Stacked', 'Cardiopulmonary Stacked', 'Renal & Hepatic Stacked'])]
#df = df[df['Modality'] != 'Invalid Modality']

df.to_excel(os.path.join(commonality_path, f'commonality_results_age_vs_brain_body_paired_male_female_RENAMED.xlsx'), index=False, engine='openpyxl')

# Display the results to verify
print("Processed Modality column values:")
print(df['Modality'].unique())